In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [3]:
X_train = joblib.load("../data/processed/X_train.pkl")
X_test = joblib.load("../data/processed/X_test.pkl")

y_train = joblib.load("../data/processed/y_train.pkl")
y_test = joblib.load("../data/processed/y_test.pkl")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (24000, 27)
X_test: (6000, 27)
y_train: (24000,)
y_test: (6000,)


In [4]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

In [5]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

    filename = name.lower().replace(" ", "_") + ".pkl"

    joblib.dump(
        model,
        f"../models/{filename}"
    )

    print(f"{name} trained and saved.")

Logistic Regression trained and saved.
Random Forest trained and saved.


In [6]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "ROC_AUC",
    ascending=False
)

display(results_df)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
1,Random Forest,0.780333,0.502905,0.587038,0.541725,0.775654
0,Logistic Regression,0.679667,0.367954,0.624717,0.463128,0.708475


In [7]:
print("Best model based on ROC-AUC:")
print(results_df.iloc[0]["Model"])

Best model based on ROC-AUC:
Random Forest
